# 04 — Traduire l'extension nécessaire à `bci_fr`

Le dataset bidirectionnel final doit utiliser chaque source une seule fois. Les 22 008 sources déjà entièrement traduites en baoulé fournissent 16 506 exemples `bci_bci` et 5 502 exemples `bci_fr`. Il manque donc 6 052 prompts baoulé pour atteindre les 11 554 exemples `bci_fr` demandés.

Ce notebook prépare et traduit **6 200 candidats** issus de l'ancienne partition `fr_fr` : 6 052 seront retenus et 148 serviront de réserve qualité. Seules les instructions et les entrées non vides sont traduites ; les réponses françaises originales sont conservées.

## 1. Sélectionner les candidats de manière déterministe

In [1]:
from collections import defaultdict
from datetime import datetime, timezone
from pathlib import Path
import asyncio
import builtins
import hashlib
import json
import os
import re
import shutil

PROJECT_ROOT = Path.cwd()
SPLIT_DIR = PROJECT_ROOT / "data" / "01_splits"
STEP_DIR_EXTENSION = PROJECT_ROOT / "data" / "03_final" / "bci_fr_extension_translation"
STEP_DIR_EXTENSION.mkdir(parents=True, exist_ok=True)
EXTENSION_SOURCE_PATH = STEP_DIR_EXTENSION / "bci_fr_extension_candidates.jsonl"
EXTENSION_MANIFEST_PATH = STEP_DIR_EXTENSION / "selection_manifest.json"
SELECTION_SEED = 20260925
REQUIRED_EXTENSION_ROWS = 6_052
RESERVE_ROWS = 148
CANDIDATE_ROWS = REQUIRED_EXTENSION_ROWS + RESERVE_ROWS

def read_jsonl_file(path):
    with path.open("r", encoding="utf-8") as handle:
        return [json.loads(line) for line in handle if line.strip()]

def write_jsonl_file_atomic(path, records):
    temporary = path.with_suffix(path.suffix + ".tmp")
    with temporary.open("w", encoding="utf-8", newline="\n") as handle:
        for record in records:
            handle.write(json.dumps(record, ensure_ascii=False) + "\n")
    temporary.replace(path)

def stable_rank(record, salt):
    value = f"{SELECTION_SEED}|{salt}|{record['id']}"
    return hashlib.sha256(value.encode("utf-8")).hexdigest()

fr_fr_source = read_jsonl_file(SPLIT_DIR / "fr_fr_source.jsonl")
ranked_fr_fr = sorted(fr_fr_source, key=lambda row: stable_rank(row, "bci_fr_extension"))
extension_candidates = ranked_fr_fr[:CANDIDATE_ROWS]
assert len(extension_candidates) == CANDIDATE_ROWS
assert len({row['id'] for row in extension_candidates}) == CANDIDATE_ROWS

write_jsonl_file_atomic(EXTENSION_SOURCE_PATH, extension_candidates)
selection_manifest = {
    "created_at_utc": datetime.now(timezone.utc).isoformat(),
    "seed": SELECTION_SEED,
    "source_partition": "fr_fr",
    "required_rows": REQUIRED_EXTENSION_ROWS,
    "reserve_rows": RESERVE_ROWS,
    "candidate_rows": len(extension_candidates),
    "candidate_ids_sha256": hashlib.sha256("\n".join(row["id"] for row in extension_candidates).encode("utf-8")).hexdigest(),
}
EXTENSION_MANIFEST_PATH.write_text(json.dumps(selection_manifest, ensure_ascii=False, indent=2) + "\n", encoding="utf-8")
print("Candidats bci_fr :", len(extension_candidates))
print("Requis             :", REQUIRED_EXTENSION_ROWS)
print("Réserve            :", RESERVE_ROWS)
print("Source             :", EXTENSION_SOURCE_PATH)

Candidats bci_fr : 6200
Requis             : 6052
Réserve            : 148
Source             : /home/vedem_worker/scrapt-bau/data/03_final/bci_fr_extension_translation/bci_fr_extension_candidates.jsonl


## 2. Préparer les unités et les blocs de traduction

In [2]:
BASE_NOTEBOOK_PATH = PROJECT_ROOT / "02b_traduire_fr_bau_texte_playwright.ipynb"
if not BASE_NOTEBOOK_PATH.exists():
    BASE_NOTEBOOK_PATH = PROJECT_ROOT / "02b_traduire_mode_texte_playwright.ipynb"
if not BASE_NOTEBOOK_PATH.exists():
    raise FileNotFoundError("Notebook moteur 02b introuvable.")

os.environ["LREE_PIPELINE_DIRECTION"] = "fr_bau"  # uniquement pour charger le moteur générique
os.environ["LREE_STEP_DIR"] = str(STEP_DIR_EXTENSION)
os.environ["LREE_DELAY_SECONDS"] = "8.0"
os.environ["LREE_INITIAL_DELAY_SECONDS"] = "0.0"

base_notebook = json.loads(BASE_NOTEBOOK_PATH.read_text(encoding="utf-8"))
base_cells = {cell.get("id"): cell for cell in base_notebook["cells"]}

def run_base_cell(cell_id):
    cell = base_cells[cell_id]
    exec(compile("".join(cell["source"]), f"{BASE_NOTEBOOK_PATH.name}:{cell_id}", "exec"), globals())

run_base_cell("cd8832ee")
PIPELINE_DIRECTION = "bci_fr"
DELAY_SECONDS = 8.0
INITIAL_DELAY_SECONDS = 0.0
read_jsonl = read_jsonl_file
write_jsonl_atomic = write_jsonl_file_atomic

def normalize_source_text(value):
    text = "" if value is None else str(value)
    text = re.sub(r"<br\s*/?>", "\n", text, flags=re.IGNORECASE)
    text = re.sub(r"&nbsp;", " ", text, flags=re.IGNORECASE).replace("\u00a0", " ")
    return text.strip()

def split_long_text(text, max_chars):
    if len(text) <= max_chars:
        return [text]
    chunks, remaining = [], text
    while len(remaining) > max_chars:
        window = remaining[:max_chars + 1]
        candidates = [window.rfind(token) for token in ("\n\n", "\n", ". ", "! ", "? ", "; ", ", ", " ")]
        cut = max(candidates)
        cut = max_chars if cut < max_chars // 2 else cut + 1
        chunks.append(remaining[:cut].strip())
        remaining = remaining[cut:].strip()
    if remaining:
        chunks.append(remaining)
    return chunks

units = []
max_payload_chars = MAX_BATCH_CHARACTERS - 80
for record in extension_candidates:
    for source_field, target_field in (("instruction_fr", "instruction_bci"), ("input_fr", "input_bci")):
        source_text = normalize_source_text(record[source_field])
        if not source_text:
            continue
        parts = split_long_text(source_text, max_payload_chars)
        for part_index, part_text in enumerate(parts):
            unit_id = f"u{len(units) + 1:08d}"
            units.append({
                "unit_id": unit_id,
                "record_id": record["id"],
                "source_row_index": record["source_row_index"],
                "direction": "bci_fr",
                "source_field": source_field,
                "target_field": target_field,
                "part_index": part_index,
                "part_count": len(parts),
                "source_text": part_text,
                "source_sha256": hashlib.sha256(part_text.encode("utf-8")).hexdigest(),
            })

write_jsonl_atomic(UNITS_PATH, units)
unit_by_id = {unit["unit_id"]: unit for unit in units}

def serialize_unit(unit):
    return f"{unit['unit_id'][1:]}: {unit['source_text']}"

batches, current_units, current_parts, current_length = [], [], [], 0
def flush_batch():
    global current_units, current_parts, current_length
    if not current_units:
        return
    source_text = "\n".join(current_parts)
    batches.append({
        "batch_id": f"batch_{len(batches) + 1:05d}",
        "unit_ids": [unit["unit_id"] for unit in current_units],
        "source_text": source_text,
        "source_characters": len(source_text),
        "source_sha256": hashlib.sha256(source_text.encode("utf-8")).hexdigest(),
    })
    current_units, current_parts, current_length = [], [], 0

for unit in units:
    serialized = serialize_unit(unit)
    separator_length = 1 if current_parts else 0
    if current_parts and current_length + separator_length + len(serialized) > MAX_BATCH_CHARACTERS:
        flush_batch()
    current_units.append(unit)
    current_parts.append(serialized)
    current_length += separator_length + len(serialized)
flush_batch()
write_jsonl_atomic(BATCHES_PATH, batches)
assert sum(len(batch["unit_ids"]) for batch in batches) == len(units)
print("Unités à traduire :", len(units))
print("Blocs préparés    :", len(batches))
print("Dossier           :", STEP_DIR)

Répertoire de travail : /home/vedem_worker/scrapt-bau/data/03_final/bci_fr_extension_translation
Direction              : fr_bau
Code cible Google     : bci
Contrôle automatique tous les blocs : 100
Unités à traduire : 8140
Blocs préparés    : 196
Dossier           : /home/vedem_worker/scrapt-bau/data/03_final/bci_fr_extension_translation


## 3. Préparer Chromium

In [3]:
run_base_cell("59834e33")
run_base_cell("1cd9faea")

Playwright est déjà installé.
Vérification/installation du navigateur Chromium compatible…
Chromium Playwright est prêt.
Bibliothèques Chromium : /home/vedem_worker/scrapt-bau/.playwright-libs/usr/lib/x86_64-linux-gnu
Existe : True


## 4. Lancer la traduction autonome

Cette cellule reprend automatiquement après une interruption, valide les résultats et remet les blocs invalides dans la file, avec trois tentatives maximum par bloc.

In [4]:
RUN_BROWSER = False
run_base_cell("d3ff9847")
RUN_BROWSER = True
retry_counts = defaultdict(int)
MAX_RETRIES_PER_BATCH = 3

def repair_extension_results(report):
    bad_batch_ids = {error.get("batch_id") for error in report.get("validation_errors", [])}
    if not bad_batch_ids:
        return
    for batch_id in bad_batch_ids:
        retry_counts[batch_id] += 1
        if retry_counts[batch_id] > MAX_RETRIES_PER_BATCH:
            raise RuntimeError(f"{batch_id} reste invalide après {MAX_RETRIES_PER_BATCH} tentatives.")
    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S_%f")
    backup_path = STEP_DIR / f"browser_results_backup_{timestamp}.jsonl"
    shutil.copy2(RESULTS_PATH, backup_path)
    records = read_jsonl(RESULTS_PATH)
    kept = [record for record in records if record.get("batch_id") not in bad_batch_ids]
    write_jsonl_atomic(RESULTS_PATH, kept)
    print("Blocs remis en traduction :", sorted(map(str, bad_batch_ids)))
    print("Sauvegarde :", backup_path)
    repaired = parse_and_validate_results(verbose=True)
    if repaired["validation_error_count"]:
        raise RuntimeError("La réparation automatique n'a pas supprimé toutes les erreurs.")

def run_extension_autonomous():
    while True:
        report = parse_and_validate_results(verbose=True)
        if report["validation_error_count"]:
            repair_extension_results(report)
        if len(completed_batch_ids()) >= len(batches):
            print("✓ Extension bci_fr entièrement traduite et validée.")
            return
        original_input = builtins.input
        builtins.input = lambda prompt="": (print(prompt + "validation automatique"), "")[1]
        try:
            run_coroutine_in_thread(run_browser_session())
        finally:
            builtins.input = original_input
        report = parse_and_validate_results(verbose=True)
        if report["validation_error_count"]:
            repair_extension_results(report)
            print("✓ Reprise automatique…")
            continue
        if len(completed_batch_ids()) >= len(batches):
            print("✓ Extension bci_fr entièrement traduite et validée.")
        else:
            print("Session interrompue sans erreur de contrôle ; relancez cette cellule.")
        return

try:
    run_extension_autonomous()
finally:
    RUN_BROWSER = False

Navigateur non lancé. Passez RUN_BROWSER à True dans la première cellule après avoir lu les consignes.
--- Contrôle automatique ---
Blocs validés       : 0/196
Fragments analysés  : 0
Champs reconstruits : 0
Erreurs de contrôle : 0
Sortie partielle     : /home/vedem_worker/scrapt-bau/data/03_final/bci_fr_extension_translation/partial_translations.jsonl
--- Contrôle automatique ---
Blocs validés       : 0/196
Fragments analysés  : 0
Champs reconstruits : 0
Erreurs de contrôle : 0
Sortie partielle     : /home/vedem_worker/scrapt-bau/data/03_final/bci_fr_extension_translation/partial_translations.jsonl
Progression initiale : 0/196 blocs terminés
Exécution continue : 196 blocs à traiter
Vérifiez Français → Baoulé dans le navigateur, puis appuyez sur Entrée ici… validation automatique
[1/196] batch_00001 — 3431 caractères
  ✓ sauvegardé
[2/196] batch_00002 — 3442 caractères
  ✓ sauvegardé
[3/196] batch_00003 — 3495 caractères
  ✓ sauvegardé
[4/196] batch_00004 — 3485 caractères
  ✓ sauvegar

## 5. Contrôle final optionnel

In [ ]:
extension_validation_report = parse_and_validate_results(verbose=True)